In [6]:
### Resample 2D slices to reference grid (retain high resolution)
from pathlib import Path
from fsl.data.image import Image
from scipy.ndimage import map_coordinates
import numpy as np

ref_file = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/Retardance/Slice_070_EnR_hdr.nii.gz')
slice_dir = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/Retardance')
output_dir = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_ngtools_test/resliced_highres_ref70')
output_dir.mkdir(exist_ok=True)

# Load reference once
ref_img_nib = Image(str(ref_file))
ref_affine = ref_img_nib.get_affine('voxel', 'world')
ref_shape = ref_img_nib.shape
if len(ref_shape) != 3:
    raise ValueError(f'Reference must be 3D, got shape={ref_shape}')

# Build reference 2D grid once (slice at x=0), then convert to world coords once
ref_2d_shape = (ref_shape[1], ref_shape[2])
y_ref, z_ref = np.meshgrid(
    np.arange(ref_2d_shape[0], dtype=np.float32),
    np.arange(ref_2d_shape[1], dtype=np.float32),
    indexing='ij'
)
x_ref = np.zeros_like(y_ref)
coords_ref_flat = np.array([x_ref, y_ref, z_ref, np.ones_like(x_ref)], dtype=np.float32).reshape(4, -1)
coords_world = ref_affine @ coords_ref_flat

# Resample each slice to reference grid
slice_files = sorted(slice_dir.glob('*.nii.gz'))
for slice_idx, slice_file in enumerate(slice_files):
    # Load 2D slice
    slice_img_nib = Image(str(slice_file))
    slice_data = np.asarray(slice_img_nib.data, dtype=np.float32).squeeze()

    # Convert world coords to this slice voxel space
    slice_affine_inv = slice_img_nib.get_affine("world", "voxel")
    coords_slice_flat = slice_affine_inv @ coords_world
    coords_slice_2d = coords_slice_flat[1:3, :].reshape(2, *ref_2d_shape)

    output_array_2d = map_coordinates(
        slice_data,
        coords_slice_2d,
        order=3,
        mode='constant',
        cval=0.0,
    ).astype(np.float32, copy=False)

    # Expand back to 3D for output (with x=0)
    output_array_3d = np.zeros(ref_shape, dtype=np.float32)
    output_array_3d[0, :, :] = output_array_2d

    output_img = Image(output_array_3d, header=ref_img_nib.header)
    output_file = output_dir / slice_file.name.replace('.nii.gz', '_resampled.nii.gz')
    output_img.save(str(output_file))


In [15]:
# Create a 3D stack from 2D resampled images
from pathlib import Path
import zarr
import numpy as np
import nibabel as nib
import time

t_start = time.time()

out_folder = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_ngtools_test')
slice_dir = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_ngtools_test/resliced_highres_ref70')
slice_files = sorted(slice_dir.glob('*.nii.gz'))

if len(slice_files) == 0:
    raise ValueError(f'No slices found in {slice_dir}')

# Load reference metadata once
ref_file = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/Retardance/Slice_070_EnR_hdr.nii.gz')
ref_img_nib = nib.load(str(ref_file))
ref_affine = ref_img_nib.affine.copy()

# Create output Zarr group
out_folder.mkdir(parents=True, exist_ok=True)
out_name = out_folder / 'vol_3D_highres_resampled_ref70_new.zarr'
root = zarr.open_group(str(out_name), mode='w', zarr_format=2)

# Input slices are stored as 3D volumes with data in the first x-plane
first_img = nib.load(str(slice_files[0]))
first_plane = np.asarray(first_img.dataobj[0, :, :], dtype=np.float32)
y, z = first_plane.shape
x = len(slice_files)

# Store as (Z, Y, X) to match axis annotation [z, y, x]
arr0 = root.create_array(
    '0',
    shape=(z, y, x),
    chunks=(min(1024, z), min(1024, y), 1),
    dtype='float32',
)

# Extract voxel size and origin
voxel_x = float(np.linalg.norm(ref_affine[:3, 0]))
voxel_y = float(np.linalg.norm(ref_affine[:3, 1]))
voxel_z = float(np.linalg.norm(ref_affine[:3, 2]))
origin = ref_affine[:3, 3].copy()

# Build diagonal affine
affine_xyz = np.array([
    [voxel_x, 0.0, 0.0, float(origin[0])],
    [0.0, voxel_y, 0.0, float(origin[1])],
    [0.0, 0.0, voxel_z, float(origin[2])],
], dtype=np.float64)

# Orientation metadata
# if Path(out_folder / 'dti_FA_in_PSOCT.zarr').exists():
#     ref_root = zarr.open_group(str(out_folder / 'dti_FA_in_PSOCT.zarr'), mode='r')
#     orientation = ref_root['nifti'].attrs.get('Orientation', {'x': 'l', 'y': 'a', 'z': 's'})
# else:
orientation = {'x': 'l', 'y': 'a', 'z': 's'}

# Initialise multiscales
max_levels = 6
xy_switch_threshold = 512
min_yz_size = 32

levels = []
cum_fx, cum_fy, cum_fz = 1, 1, 1
cur_x, cur_y, cur_z = x, y, z

for level_idx in range(1, max_levels + 1):
    if min(cur_y, cur_z) < min_yz_size:
        break

    use_x_downsample = (min(cur_y, cur_z) <= xy_switch_threshold) and (cur_x >= 2)
    step_fx = 2 if use_x_downsample else 1
    step_fy = 2 if cur_y >= 2 else 1
    step_fz = 2 if cur_z >= 2 else 1

    if (step_fx, step_fy, step_fz) == (1, 1, 1):
        break

    cum_fx *= step_fx
    cum_fy *= step_fy
    cum_fz *= step_fz

    out_x = x // cum_fx
    out_y = y // cum_fy
    out_z = z // cum_fz
    if out_x == 0 or out_y == 0 or out_z == 0:
        break

    level_name = str(level_idx)
    level_array = root.create_array(
        level_name,
        shape=(out_z, out_y, out_x),
        chunks=(min(512, out_z), min(512, out_y), 1),
        dtype='float32',
    )

    levels.append({
        'name': level_name,
        'array': level_array,
        'cum_fx': cum_fx,
        'cum_fy': cum_fy,
        'cum_fz': cum_fz,
        'out_x': out_x,
        'out_y': out_y,
        'out_z': out_z,
        'x_trim': out_x * cum_fx,
        'y_trim': out_y * cum_fy,
        'z_trim': out_z * cum_fz,
        'buffer': np.zeros((out_z, out_y), dtype=np.float32),
        'buffer_count': 0,
        'x_out_idx': 0,
    })

    cur_x, cur_y, cur_z = out_x, out_y, out_z

t_end_init = time.time()
print("Initialisation time", t_end_init - t_start)

# ==== Main loop: load slices and update zarr ====
overall_min = float('inf')
overall_max = float('-inf')

for i, p in enumerate(slice_files):
    t_start_slice = time.time()

    img = nib.load(str(p))
    data_yz = np.asarray(img.dataobj[0, :, :], dtype=np.float32)

    if data_yz.shape != (y, z):
        raise ValueError(f'shape mismatch for {p.name}: {data_yz.shape} != {(y, z)}')

    # Store directly to Zarr: data is (Y, Z), store transposed to arr0[Z, Y, X]
    arr0[:, :, i] = data_yz.T
    
    t_end_highres = time.time()
    print(f"Writing highres time - slice {i}", t_end_highres - t_end_init)

    # Track min/max
    smin = float(data_yz.min())
    smax = float(data_yz.max())
    if smin < overall_min:
        overall_min = smin
    if smax > overall_max:
        overall_max = smax

    # Write pyramid levels from the same slice
    for lvl in levels:
        if i >= lvl['x_trim']:
            continue

        d2 = data_yz[:lvl['y_trim'], :lvl['z_trim']]
        yz_down = d2.reshape(
            lvl['out_y'], lvl['cum_fy'], lvl['out_z'], lvl['cum_fz']
        ).mean(axis=(1, 3), dtype=np.float32)

        if lvl['cum_fx'] == 1:
            lvl['array'][:, :, i] = yz_down.T
        else:
            lvl['buffer'] += yz_down.T
            lvl['buffer_count'] += 1
            if lvl['buffer_count'] == lvl['cum_fx']:
                lvl['array'][:, :, lvl['x_out_idx']] = (lvl['buffer'] / lvl['cum_fx']).astype(np.float32, copy=False)
                lvl['x_out_idx'] += 1
                lvl['buffer'].fill(0)
                lvl['buffer_count'] = 0

    t_end_lowres = time.time()
    print(f"Writing multiscales time - slice {i}", t_end_lowres - t_end_highres)

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(slice_files)} files")

# Metadata
root.attrs['data_min'] = float(overall_min)
root.attrs['data_max'] = float(overall_max)

datasets = []
for idx in range(0, len(levels) + 1):
    if idx == 0:
        fx = fy = fz = 1
        path = '0'
    else:
        lvl = levels[idx - 1]
        fx, fy, fz = lvl['cum_fx'], lvl['cum_fy'], lvl['cum_fz']
        path = lvl['name']

    sx = voxel_x * fx
    sy = voxel_y * fy
    sz = voxel_z * fz

    tx = 0.5 * (fx - 1) * voxel_x
    ty = 0.5 * (fy - 1) * voxel_y
    tz = 0.5 * (fz - 1) * voxel_z

    datasets.append({
        'path': path,
        'coordinateTransformations': [
            {'type': 'scale', 'scale': [sz, sy, sx]},
            {'type': 'translation', 'translation': [tz, ty, tx]},
        ],
    })

root.attrs['multiscales'] = [{
    'axes': [
        {'name': 'z', 'type': 'space', 'unit': 'millimeter'},
        {'name': 'y', 'type': 'space', 'unit': 'millimeter'},
        {'name': 'x', 'type': 'space', 'unit': 'millimeter'},
    ],
    'coordinateTransformations': [
        {'type': 'scale', 'scale': [1.0, 1.0, 1.0]}
    ],
    'datasets': datasets,
    'name': '',
    'type': 'median window 2x2x2',
    'version': '0.4',
}]

# Store nifti metadata
nifti_group = root.create_group('nifti')
nifti_group.create_array('0', shape=(348,), chunks=(348,), dtype='uint8')

nifti_group.attrs.update({
    'Affine': affine_xyz[:3, :4].tolist(),
    'Dim': [int(x), int(y), int(z)],
    'VoxelSize': [voxel_x, voxel_y, voxel_z],
    'Orientation': orientation,
    'SForm': 'aligned_anat',
    'QForm': '',
    'QuaternOffset': {
        'x': float(origin[0]),
        'y': float(origin[1]),
        'z': float(origin[2]),
    },
    'Unit': {'L': 'mm', 'T': 's'},
    'DataType': 'single',
    'BitDepth': 32,
    'NIIHeaderSize': 348,
})
t_end = time.time()
print("Metadata time", t_end - t_end_lowres)



Initialisation time 2.014446973800659
Writing highres time - slice 0 0.712230920791626
Writing multiscales time - slice 0 0.8006551265716553
Writing highres time - slice 1 2.296092987060547
Writing multiscales time - slice 1 0.7419719696044922
Writing highres time - slice 2 3.6770730018615723
Writing multiscales time - slice 2 0.7615158557891846
Writing highres time - slice 3 5.096124887466431
Writing multiscales time - slice 3 0.7699480056762695
Writing highres time - slice 4 6.5295729637146
Writing multiscales time - slice 4 0.751338005065918
Writing highres time - slice 5 7.937201976776123
Writing multiscales time - slice 5 0.7747149467468262
Writing highres time - slice 6 9.43191409111023
Writing multiscales time - slice 6 0.7841498851776123
Writing highres time - slice 7 11.131947040557861
Writing multiscales time - slice 7 0.8110659122467041
Writing highres time - slice 8 12.858873844146729
Writing multiscales time - slice 8 0.7956490516662598
Writing highres time - slice 9 14.50